# Unnormalized Neural–DTB Game Dynamics

This notebook clones the project from GitHub, verifies the implementation, and runs linear and Cournot examples. Choose a GPU runtime if available.

## 1. Clone the GitHub branch

Colab clones GitHub into temporary runtime storage. After the feature branch is merged, change `BRANCH` to `main`.

In [ ]:
import pathlib, shutil, subprocess

REPO_URL = 'https://github.com/sun-mengwei/dtb-colab-experiments.git'
BRANCH = 'codex/game-dynamics-dtb'  # use 'main' after merge
REPO_DIR = pathlib.Path('/content/dtb-colab-experiments')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    REPO_URL, str(REPO_DIR),
], check=True)

PROJECT_DIR = REPO_DIR / 'dtb_game_dynamics_unnormalized'
assert (PROJECT_DIR / 'run_game_dynamics.py').exists(), PROJECT_DIR

In [ ]:
%cd /content/dtb-colab-experiments/dtb_game_dynamics_unnormalized
!python -m pip install -q -r requirements.txt

In [ ]:
import platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Verify the implementation

These tests cover unnormalized stacking, the SVD projection, spatial derivatives, Gaussian initialization, and a complete zero-velocity step.

In [ ]:
!python -m pytest

## 3. Linear-quadratic smoke experiment

In [ ]:
!python run_game_dynamics.py --game linear --particles 32 --steps 5 --basis-size 32 --width 12 --depth 2 --dtype float32 --device auto --output-dir outputs/colab_linear

In [ ]:
from IPython.display import Image, display
display(Image('outputs/colab_linear/summary.png'))

In [ ]:
import numpy as np
data = np.load('outputs/colab_linear/history.npz')
print('final mean:', data['means'][-1])
print('projection residuals:', data['projection_residuals'])
print('retained ranks:', data['retained_ranks'])
print('alpha norms:', data['alpha_norms'])
print('final score RMS:', np.sqrt(np.mean(data['final_score'] ** 2)))

## 4. Cournot experiment

Start small because the score update uses nested automatic differentiation.

In [ ]:
!python run_game_dynamics.py --game cournot --particles 64 --steps 10 --step-size 0.005 --basis-size 64 --svd-rtol 1e-5 --diffusion 0.03 --width 24 --depth 2 --dtype float32 --device auto --output-dir outputs/colab_cournot

In [ ]:
display(Image('outputs/colab_cournot/summary.png'))

## 5. Save results to Google Drive

GitHub stores the source. Mount Drive to preserve runtime outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!zip -qr dtb_game_outputs.zip outputs
!cp dtb_game_outputs.zip '/content/drive/MyDrive/'